# 01 Data Integration

## Goal
Integrate raw geriatric healthcare tables into a patient-level analytics dataset for downstream care gap analysis, segmentation, and predictive modeling.

## 1. Load raw healthcare tables
2: Import libraries


In [100]:
import pandas as pd
import numpy as np
from pathlib import Path

3: Set file paths

In [101]:
DATA_DIR = Path("../data/raw")

patients = pd.read_csv(DATA_DIR / "patients.csv")
encounters = pd.read_csv(DATA_DIR / "encounters.csv")
conditions = pd.read_csv(DATA_DIR / "conditions.csv")
medications = pd.read_csv(DATA_DIR / "medications.csv")
claims = pd.read_csv(DATA_DIR / "claims.csv")
observations = pd.read_csv(DATA_DIR / "observations.csv")
providers = pd.read_csv(DATA_DIR / "providers.csv")

## 2. Inspect schema and join keys

In [102]:
# inspecting pateints file
patients.head()
patients.shape
patients.columns

Index(['Id', 'BIRTHDATE', 'DEATHDATE', 'SSN', 'DRIVERS', 'PASSPORT', 'PREFIX',
       'FIRST', 'MIDDLE', 'LAST', 'SUFFIX', 'MAIDEN', 'MARITAL', 'RACE',
       'ETHNICITY', 'GENDER', 'BIRTHPLACE', 'ADDRESS', 'CITY', 'STATE',
       'COUNTY', 'FIPS', 'ZIP', 'LAT', 'LON', 'HEALTHCARE_EXPENSES',
       'HEALTHCARE_COVERAGE', 'INCOME'],
      dtype='object')

In [103]:
# inspecting encounters file
encounters.head()
encounters.shape
encounters.columns

Index(['Id', 'START', 'STOP', 'PATIENT', 'ORGANIZATION', 'PROVIDER', 'PAYER',
       'ENCOUNTERCLASS', 'CODE', 'DESCRIPTION', 'BASE_ENCOUNTER_COST',
       'TOTAL_CLAIM_COST', 'PAYER_COVERAGE', 'REASONCODE',
       'REASONDESCRIPTION'],
      dtype='object')

In [104]:
# inspecting conditions, observations, claims,medications files
conditions.head()
conditions.shape
conditions.columns

Index(['START', 'STOP', 'PATIENT', 'ENCOUNTER', 'SYSTEM', 'CODE',
       'DESCRIPTION'],
      dtype='object')

In [105]:
# inspecting conditions, conditions, claims,medications files
claims.head()
claims.shape
claims.columns

Index(['Id', 'PATIENTID', 'PROVIDERID', 'PRIMARYPATIENTINSURANCEID',
       'SECONDARYPATIENTINSURANCEID', 'DEPARTMENTID', 'PATIENTDEPARTMENTID',
       'DIAGNOSIS1', 'DIAGNOSIS2', 'DIAGNOSIS3', 'DIAGNOSIS4', 'DIAGNOSIS5',
       'DIAGNOSIS6', 'DIAGNOSIS7', 'DIAGNOSIS8', 'REFERRINGPROVIDERID',
       'APPOINTMENTID', 'CURRENTILLNESSDATE', 'SERVICEDATE',
       'SUPERVISINGPROVIDERID', 'STATUS1', 'STATUS2', 'STATUSP',
       'OUTSTANDING1', 'OUTSTANDING2', 'OUTSTANDINGP', 'LASTBILLEDDATE1',
       'LASTBILLEDDATE2', 'LASTBILLEDDATEP', 'HEALTHCARECLAIMTYPEID1',
       'HEALTHCARECLAIMTYPEID2'],
      dtype='object')

In [106]:
medications.head()
medications.shape
medications.columns

Index(['START', 'STOP', 'PATIENT', 'PAYER', 'ENCOUNTER', 'CODE', 'DESCRIPTION',
       'BASE_COST', 'PAYER_COVERAGE', 'DISPENSES', 'TOTALCOST', 'REASONCODE',
       'REASONDESCRIPTION'],
      dtype='object')

In [107]:

observations.head()
observations.shape
observations.columns

Index(['DATE', 'PATIENT', 'ENCOUNTER', 'CATEGORY', 'CODE', 'DESCRIPTION',
       'VALUE', 'UNITS', 'TYPE'],
      dtype='object')

5: Identify the key column for joining

## 🔗 Join Key

The primary join key across tables is the patient identifier, which appears as:

- `Id` in patients.csv  
- `PATIENT` in encounters, conditions, medications, observations  
- `PATIENTID` in claims.csv  
This key is used to link patient-level data across all tables.


In [108]:
#loading tables again
DATA_DIR = Path("../data/raw")

patients = pd.read_csv(DATA_DIR / "patients.csv")
encounters = pd.read_csv(DATA_DIR / "encounters.csv")
conditions = pd.read_csv(DATA_DIR / "conditions.csv")
medications = pd.read_csv(DATA_DIR / "medications.csv")
claims = pd.read_csv(DATA_DIR / "claims.csv")
observations = pd.read_csv(DATA_DIR / "observations.csv")
providers = pd.read_csv(DATA_DIR / "providers.csv")

In [109]:
#Standardize join keys
patients = patients.rename(columns={"Id": "patient_id"})
encounters = encounters.rename(columns={"PATIENT": "patient_id", "Id": "encounter_id"})
conditions = conditions.rename(columns={"PATIENT": "patient_id", "ENCOUNTER": "encounter_id"})
medications = medications.rename(columns={"PATIENT": "patient_id", "ENCOUNTER": "encounter_id"})
observations = observations.rename(columns={"PATIENT": "patient_id", "ENCOUNTER": "encounter_id"})
claims = claims.rename(columns={"PATIENTID": "patient_id"})

In [110]:
print("patients:", patients.columns.tolist())
print("encounters:", encounters.columns.tolist())
print("conditions:", conditions.columns.tolist())
print("medications:", medications.columns.tolist())


patients: ['patient_id', 'BIRTHDATE', 'DEATHDATE', 'SSN', 'DRIVERS', 'PASSPORT', 'PREFIX', 'FIRST', 'MIDDLE', 'LAST', 'SUFFIX', 'MAIDEN', 'MARITAL', 'RACE', 'ETHNICITY', 'GENDER', 'BIRTHPLACE', 'ADDRESS', 'CITY', 'STATE', 'COUNTY', 'FIPS', 'ZIP', 'LAT', 'LON', 'HEALTHCARE_EXPENSES', 'HEALTHCARE_COVERAGE', 'INCOME']
encounters: ['encounter_id', 'START', 'STOP', 'patient_id', 'ORGANIZATION', 'PROVIDER', 'PAYER', 'ENCOUNTERCLASS', 'CODE', 'DESCRIPTION', 'BASE_ENCOUNTER_COST', 'TOTAL_CLAIM_COST', 'PAYER_COVERAGE', 'REASONCODE', 'REASONDESCRIPTION']
conditions: ['START', 'STOP', 'patient_id', 'encounter_id', 'SYSTEM', 'CODE', 'DESCRIPTION']
medications: ['START', 'STOP', 'patient_id', 'PAYER', 'encounter_id', 'CODE', 'DESCRIPTION', 'BASE_COST', 'PAYER_COVERAGE', 'DISPENSES', 'TOTALCOST', 'REASONCODE', 'REASONDESCRIPTION']


In [111]:
#rerunning the clean code
patients = patients.rename(columns={"Id": "patient_id"})
encounters = encounters.rename(columns={"PATIENT": "patient_id", "Id": "encounter_id"})
conditions = conditions.rename(columns={"PATIENT": "patient_id", "ENCOUNTER": "encounter_id"})
medications = medications.rename(columns={"PATIENT": "patient_id", "ENCOUNTER": "encounter_id"})
observations = observations.rename(columns={"PATIENT": "patient_id", "ENCOUNTER": "encounter_id"})
claims = claims.rename(columns={"PATIENTID": "patient_id"})


In [113]:
print("patients:", patients.columns.tolist())
print("encounters:", encounters.columns.tolist())
print("medications:", medications.columns.tolist())


patients: ['patient_id', 'BIRTHDATE', 'DEATHDATE', 'SSN', 'DRIVERS', 'PASSPORT', 'PREFIX', 'FIRST', 'MIDDLE', 'LAST', 'SUFFIX', 'MAIDEN', 'MARITAL', 'RACE', 'ETHNICITY', 'GENDER', 'BIRTHPLACE', 'ADDRESS', 'CITY', 'STATE', 'COUNTY', 'FIPS', 'ZIP', 'LAT', 'LON', 'HEALTHCARE_EXPENSES', 'HEALTHCARE_COVERAGE', 'INCOME']
encounters: ['encounter_id', 'START', 'STOP', 'patient_id', 'ORGANIZATION', 'PROVIDER', 'PAYER', 'ENCOUNTERCLASS', 'CODE', 'DESCRIPTION', 'BASE_ENCOUNTER_COST', 'TOTAL_CLAIM_COST', 'PAYER_COVERAGE', 'REASONCODE', 'REASONDESCRIPTION']
medications: ['START', 'STOP', 'patient_id', 'PAYER', 'encounter_id', 'CODE', 'DESCRIPTION', 'BASE_COST', 'PAYER_COVERAGE', 'DISPENSES', 'TOTALCOST', 'REASONCODE', 'REASONDESCRIPTION']


In [114]:
print(medications.columns)
print(encounters.columns)
print(patients.columns)


Index(['START', 'STOP', 'patient_id', 'PAYER', 'encounter_id', 'CODE',
       'DESCRIPTION', 'BASE_COST', 'PAYER_COVERAGE', 'DISPENSES', 'TOTALCOST',
       'REASONCODE', 'REASONDESCRIPTION'],
      dtype='object')
Index(['encounter_id', 'START', 'STOP', 'patient_id', 'ORGANIZATION',
       'PROVIDER', 'PAYER', 'ENCOUNTERCLASS', 'CODE', 'DESCRIPTION',
       'BASE_ENCOUNTER_COST', 'TOTAL_CLAIM_COST', 'PAYER_COVERAGE',
       'REASONCODE', 'REASONDESCRIPTION'],
      dtype='object')
Index(['patient_id', 'BIRTHDATE', 'DEATHDATE', 'SSN', 'DRIVERS', 'PASSPORT',
       'PREFIX', 'FIRST', 'MIDDLE', 'LAST', 'SUFFIX', 'MAIDEN', 'MARITAL',
       'RACE', 'ETHNICITY', 'GENDER', 'BIRTHPLACE', 'ADDRESS', 'CITY', 'STATE',
       'COUNTY', 'FIPS', 'ZIP', 'LAT', 'LON', 'HEALTHCARE_EXPENSES',
       'HEALTHCARE_COVERAGE', 'INCOME'],
      dtype='object')


In [115]:
test = medications.merge(encounters[["encounter_id", "patient_id"]], on="encounter_id", how="left")
print(test.columns)


Index(['START', 'STOP', 'patient_id_x', 'PAYER', 'encounter_id', 'CODE',
       'DESCRIPTION', 'BASE_COST', 'PAYER_COVERAGE', 'DISPENSES', 'TOTALCOST',
       'REASONCODE', 'REASONDESCRIPTION', 'patient_id_y'],
      dtype='object')


In [116]:
#rename pateint columns
medications = medications.rename(columns={"patient_id": "patient_id_med"})
encounters = encounters.rename(columns={"patient_id": "patient_id_enc"})
patients = patients.rename(columns={"patient_id": "patient_id_pat"})


In [117]:
#explicit merge of medications
medications_merged = medications.merge(
    encounters[["encounter_id", "patient_id_enc"]],
    on="encounter_id",
    how="left"
).merge(
    patients,
    left_on="patient_id_enc",
    right_on="patient_id_pat",
    how="left"
)


In [118]:
#Merge conditions → encounters → patients
conditions_merged = conditions.merge(
    encounters[["encounter_id", "patient_id_enc"]],
    on="encounter_id",
    how="left"
).merge(
    patients,
    left_on="patient_id_enc",
    right_on="patient_id_pat",
    how="left"
)


In [119]:
#Merge observations → encounters → patients
observations_merged = observations.merge(
    encounters[["encounter_id", "patient_id_enc"]],
    on="encounter_id",
    how="left"
).merge(
    patients,
    left_on="patient_id_enc",
    right_on="patient_id_pat",
    how="left"
)


In [120]:
#Merge claims → patients
claims_merged = claims.merge(
    patients,
    left_on="patient_id",
    right_on="patient_id_pat",
    how="left"
)


In [121]:
#verify each merged table
print(medications_merged.columns)
medications_merged.head()

Index(['START', 'STOP', 'patient_id_med', 'PAYER', 'encounter_id', 'CODE',
       'DESCRIPTION', 'BASE_COST', 'PAYER_COVERAGE', 'DISPENSES', 'TOTALCOST',
       'REASONCODE', 'REASONDESCRIPTION', 'patient_id_enc', 'patient_id_pat',
       'BIRTHDATE', 'DEATHDATE', 'SSN', 'DRIVERS', 'PASSPORT', 'PREFIX',
       'FIRST', 'MIDDLE', 'LAST', 'SUFFIX', 'MAIDEN', 'MARITAL', 'RACE',
       'ETHNICITY', 'GENDER', 'BIRTHPLACE', 'ADDRESS', 'CITY', 'STATE',
       'COUNTY', 'FIPS', 'ZIP', 'LAT', 'LON', 'HEALTHCARE_EXPENSES',
       'HEALTHCARE_COVERAGE', 'INCOME'],
      dtype='object')


,START,STOP,patient_id_med,PAYER,encounter_id,CODE,DESCRIPTION,BASE_COST,PAYER_COVERAGE,DISPENSES,...,CITY,STATE,COUNTY,FIPS,ZIP,LAT,LON,HEALTHCARE_EXPENSES,HEALTHCARE_COVERAGE,INCOME
0,2023-03-12T17:08:24Z,2023-04-22T14:36:55Z,146ca7c8-be75-f1bc-9e9b-c56a5caa5842,26aab0cd-6aba-3e1b-ac5b-05c8867e762c,146ca7c8-be75-f1bc-1ce8-9ba5301d7837,861467,Meperidine Hydrochloride 50 MG Oral Tablet,17.12,0.0,2,...,East Longmeadow,Massachusetts,Hampden County,NaN,0,42.028287,-72.497059,74899.55,10475.60,876903
1,2023-03-12T17:08:24Z,2023-06-06T17:08:24Z,146ca7c8-be75-f1bc-9e9b-c56a5caa5842,26aab0cd-6aba-3e1b-ac5b-05c8867e762c,146ca7c8-be75-f1bc-1ce8-9ba5301d7837,849574,Naproxen sodium 220 MG Oral Tablet,157.75,0.0,2,...,East Longmeadow,Massachusetts,Hampden County,NaN,0,42.028287,-72.497059,74899.55,10475.60,876903
2,2024-03-23T17:56:24Z,2024-03-23T17:56:24Z,146ca7c8-be75-f1bc-9e9b-c56a5caa5842,26aab0cd-6aba-3e1b-ac5b-05c8867e762c,146ca7c8-be75-f1bc-a4a3-62fc1ef2113f,1535362,sodium fluoride 0.0272 MG/MG Oral Gel,129.94,0.0,1,...,East Longmeadow,Massachusetts,Hampden County,NaN,0,42.028287,-72.497059,74899.55,10475.60,876903
3,2017-03-25T23:07:15Z,2017-04-13T23:07:15Z,5b0065a1-f616-5871-5939-075d1006d277,26aab0cd-6aba-3e1b-ac5b-05c8867e762c,5b0065a1-f616-5871-1759-c4585ad054c8,849574,Naproxen sodium 220 MG Oral Tablet,91.72,0.0,1,...,Malden,Massachusetts,Middlesex County,25017.0,2155,42.454511,-71.124463,138979.05,55016.58,808564
4,2023-03-10T16:39:17Z,2023-03-10T16:39:17Z,5b0065a1-f616-5871-5939-075d1006d277,26aab0cd-6aba-3e1b-ac5b-05c8867e762c,5b0065a1-f616-5871-d775-4ad52f6d9bf5,1535362,sodium fluoride 0.0272 MG/MG Oral Gel,129.94,0.0,1,...,Malden,Massachusetts,Middlesex County,25017.0,2155,42.454511,-71.124463,138979.05,55016.58,808564


In [122]:
print(conditions_merged.columns)
conditions_merged.head()


Index(['START', 'STOP', 'patient_id', 'encounter_id', 'SYSTEM', 'CODE',
       'DESCRIPTION', 'patient_id_enc', 'patient_id_pat', 'BIRTHDATE',
       'DEATHDATE', 'SSN', 'DRIVERS', 'PASSPORT', 'PREFIX', 'FIRST', 'MIDDLE',
       'LAST', 'SUFFIX', 'MAIDEN', 'MARITAL', 'RACE', 'ETHNICITY', 'GENDER',
       'BIRTHPLACE', 'ADDRESS', 'CITY', 'STATE', 'COUNTY', 'FIPS', 'ZIP',
       'LAT', 'LON', 'HEALTHCARE_EXPENSES', 'HEALTHCARE_COVERAGE', 'INCOME'],
      dtype='object')


,START,STOP,patient_id,encounter_id,SYSTEM,CODE,DESCRIPTION,patient_id_enc,patient_id_pat,BIRTHDATE,...,CITY,STATE,COUNTY,FIPS,ZIP,LAT,LON,HEALTHCARE_EXPENSES,HEALTHCARE_COVERAGE,INCOME
0,2014-02-22,NaN,146ca7c8-be75-f1bc-9e9b-c56a5caa5842,146ca7c8-be75-f1bc-9d2c-76218dbf93cd,http://snomed.info/sct,224299000,Received higher education (finding),146ca7c8-be75-f1bc-9e9b-c56a5caa5842,146ca7c8-be75-f1bc-9e9b-c56a5caa5842,1995-12-30,...,East Longmeadow,Massachusetts,Hampden County,NaN,0,42.028287,-72.497059,74899.55,10475.6,876903
1,2014-02-22,2018-03-03,146ca7c8-be75-f1bc-9e9b-c56a5caa5842,146ca7c8-be75-f1bc-9d2c-76218dbf93cd,http://snomed.info/sct,160904001,Part-time employment (finding),146ca7c8-be75-f1bc-9e9b-c56a5caa5842,146ca7c8-be75-f1bc-9e9b-c56a5caa5842,1995-12-30,...,East Longmeadow,Massachusetts,Hampden County,NaN,0,42.028287,-72.497059,74899.55,10475.6,876903
2,2015-02-28,2024-03-09,146ca7c8-be75-f1bc-9e9b-c56a5caa5842,146ca7c8-be75-f1bc-3354-c81e8093a56e,http://snomed.info/sct,73595000,Stress (finding),146ca7c8-be75-f1bc-9e9b-c56a5caa5842,146ca7c8-be75-f1bc-9e9b-c56a5caa5842,1995-12-30,...,East Longmeadow,Massachusetts,Hampden County,NaN,0,42.028287,-72.497059,74899.55,10475.6,876903
3,2018-03-03,2018-03-03,146ca7c8-be75-f1bc-9e9b-c56a5caa5842,146ca7c8-be75-f1bc-44b6-776d457b75c0,http://snomed.info/sct,314529007,Medication review due (situation),146ca7c8-be75-f1bc-9e9b-c56a5caa5842,146ca7c8-be75-f1bc-9e9b-c56a5caa5842,1995-12-30,...,East Longmeadow,Massachusetts,Hampden County,NaN,0,42.028287,-72.497059,74899.55,10475.6,876903
4,2018-03-03,2021-03-06,146ca7c8-be75-f1bc-9e9b-c56a5caa5842,146ca7c8-be75-f1bc-44b6-776d457b75c0,http://snomed.info/sct,160903007,Full-time employment (finding),146ca7c8-be75-f1bc-9e9b-c56a5caa5842,146ca7c8-be75-f1bc-9e9b-c56a5caa5842,1995-12-30,...,East Longmeadow,Massachusetts,Hampden County,NaN,0,42.028287,-72.497059,74899.55,10475.6,876903


In [123]:
print(observations_merged.columns)
observations_merged.head()


Index(['DATE', 'patient_id', 'encounter_id', 'CATEGORY', 'CODE', 'DESCRIPTION',
       'VALUE', 'UNITS', 'TYPE', 'patient_id_enc', 'patient_id_pat',
       'BIRTHDATE', 'DEATHDATE', 'SSN', 'DRIVERS', 'PASSPORT', 'PREFIX',
       'FIRST', 'MIDDLE', 'LAST', 'SUFFIX', 'MAIDEN', 'MARITAL', 'RACE',
       'ETHNICITY', 'GENDER', 'BIRTHPLACE', 'ADDRESS', 'CITY', 'STATE',
       'COUNTY', 'FIPS', 'ZIP', 'LAT', 'LON', 'HEALTHCARE_EXPENSES',
       'HEALTHCARE_COVERAGE', 'INCOME'],
      dtype='object')


,DATE,patient_id,encounter_id,CATEGORY,CODE,DESCRIPTION,VALUE,UNITS,TYPE,patient_id_enc,...,CITY,STATE,COUNTY,FIPS,ZIP,LAT,LON,HEALTHCARE_EXPENSES,HEALTHCARE_COVERAGE,INCOME
0,2018-03-03T14:36:55Z,146ca7c8-be75-f1bc-9e9b-c56a5caa5842,146ca7c8-be75-f1bc-44b6-776d457b75c0,vital-signs,8302-2,Body Height,170.0,cm,numeric,146ca7c8-be75-f1bc-9e9b-c56a5caa5842,...,East Longmeadow,Massachusetts,Hampden County,NaN,0.0,42.028287,-72.497059,74899.55,10475.6,876903.0
1,2018-03-03T14:36:55Z,146ca7c8-be75-f1bc-9e9b-c56a5caa5842,146ca7c8-be75-f1bc-44b6-776d457b75c0,vital-signs,72514-3,Pain severity - 0-10 verbal numeric rating [Sc...,4.0,{score},numeric,146ca7c8-be75-f1bc-9e9b-c56a5caa5842,...,East Longmeadow,Massachusetts,Hampden County,NaN,0.0,42.028287,-72.497059,74899.55,10475.6,876903.0
2,2018-03-03T14:36:55Z,146ca7c8-be75-f1bc-9e9b-c56a5caa5842,146ca7c8-be75-f1bc-44b6-776d457b75c0,vital-signs,29463-7,Body Weight,78.7,kg,numeric,146ca7c8-be75-f1bc-9e9b-c56a5caa5842,...,East Longmeadow,Massachusetts,Hampden County,NaN,0.0,42.028287,-72.497059,74899.55,10475.6,876903.0
3,2018-03-03T14:36:55Z,146ca7c8-be75-f1bc-9e9b-c56a5caa5842,146ca7c8-be75-f1bc-44b6-776d457b75c0,vital-signs,39156-5,Body mass index (BMI) [Ratio],27.2,kg/m2,numeric,146ca7c8-be75-f1bc-9e9b-c56a5caa5842,...,East Longmeadow,Massachusetts,Hampden County,NaN,0.0,42.028287,-72.497059,74899.55,10475.6,876903.0
4,2018-03-03T14:36:55Z,146ca7c8-be75-f1bc-9e9b-c56a5caa5842,146ca7c8-be75-f1bc-44b6-776d457b75c0,vital-signs,8462-4,Diastolic Blood Pressure,63.0,mm[Hg],numeric,146ca7c8-be75-f1bc-9e9b-c56a5caa5842,...,East Longmeadow,Massachusetts,Hampden County,NaN,0.0,42.028287,-72.497059,74899.55,10475.6,876903.0


In [124]:
print(observations_merged.columns)
observations_merged.head()


Index(['DATE', 'patient_id', 'encounter_id', 'CATEGORY', 'CODE', 'DESCRIPTION',
       'VALUE', 'UNITS', 'TYPE', 'patient_id_enc', 'patient_id_pat',
       'BIRTHDATE', 'DEATHDATE', 'SSN', 'DRIVERS', 'PASSPORT', 'PREFIX',
       'FIRST', 'MIDDLE', 'LAST', 'SUFFIX', 'MAIDEN', 'MARITAL', 'RACE',
       'ETHNICITY', 'GENDER', 'BIRTHPLACE', 'ADDRESS', 'CITY', 'STATE',
       'COUNTY', 'FIPS', 'ZIP', 'LAT', 'LON', 'HEALTHCARE_EXPENSES',
       'HEALTHCARE_COVERAGE', 'INCOME'],
      dtype='object')


,DATE,patient_id,encounter_id,CATEGORY,CODE,DESCRIPTION,VALUE,UNITS,TYPE,patient_id_enc,...,CITY,STATE,COUNTY,FIPS,ZIP,LAT,LON,HEALTHCARE_EXPENSES,HEALTHCARE_COVERAGE,INCOME
0,2018-03-03T14:36:55Z,146ca7c8-be75-f1bc-9e9b-c56a5caa5842,146ca7c8-be75-f1bc-44b6-776d457b75c0,vital-signs,8302-2,Body Height,170.0,cm,numeric,146ca7c8-be75-f1bc-9e9b-c56a5caa5842,...,East Longmeadow,Massachusetts,Hampden County,NaN,0.0,42.028287,-72.497059,74899.55,10475.6,876903.0
1,2018-03-03T14:36:55Z,146ca7c8-be75-f1bc-9e9b-c56a5caa5842,146ca7c8-be75-f1bc-44b6-776d457b75c0,vital-signs,72514-3,Pain severity - 0-10 verbal numeric rating [Sc...,4.0,{score},numeric,146ca7c8-be75-f1bc-9e9b-c56a5caa5842,...,East Longmeadow,Massachusetts,Hampden County,NaN,0.0,42.028287,-72.497059,74899.55,10475.6,876903.0
2,2018-03-03T14:36:55Z,146ca7c8-be75-f1bc-9e9b-c56a5caa5842,146ca7c8-be75-f1bc-44b6-776d457b75c0,vital-signs,29463-7,Body Weight,78.7,kg,numeric,146ca7c8-be75-f1bc-9e9b-c56a5caa5842,...,East Longmeadow,Massachusetts,Hampden County,NaN,0.0,42.028287,-72.497059,74899.55,10475.6,876903.0
3,2018-03-03T14:36:55Z,146ca7c8-be75-f1bc-9e9b-c56a5caa5842,146ca7c8-be75-f1bc-44b6-776d457b75c0,vital-signs,39156-5,Body mass index (BMI) [Ratio],27.2,kg/m2,numeric,146ca7c8-be75-f1bc-9e9b-c56a5caa5842,...,East Longmeadow,Massachusetts,Hampden County,NaN,0.0,42.028287,-72.497059,74899.55,10475.6,876903.0
4,2018-03-03T14:36:55Z,146ca7c8-be75-f1bc-9e9b-c56a5caa5842,146ca7c8-be75-f1bc-44b6-776d457b75c0,vital-signs,8462-4,Diastolic Blood Pressure,63.0,mm[Hg],numeric,146ca7c8-be75-f1bc-9e9b-c56a5caa5842,...,East Longmeadow,Massachusetts,Hampden County,NaN,0.0,42.028287,-72.497059,74899.55,10475.6,876903.0


In [125]:
print(claims_merged.columns)
claims_merged.head()


Index(['Id', 'patient_id', 'PROVIDERID', 'PRIMARYPATIENTINSURANCEID',
       'SECONDARYPATIENTINSURANCEID', 'DEPARTMENTID', 'PATIENTDEPARTMENTID',
       'DIAGNOSIS1', 'DIAGNOSIS2', 'DIAGNOSIS3', 'DIAGNOSIS4', 'DIAGNOSIS5',
       'DIAGNOSIS6', 'DIAGNOSIS7', 'DIAGNOSIS8', 'REFERRINGPROVIDERID',
       'APPOINTMENTID', 'CURRENTILLNESSDATE', 'SERVICEDATE',
       'SUPERVISINGPROVIDERID', 'STATUS1', 'STATUS2', 'STATUSP',
       'OUTSTANDING1', 'OUTSTANDING2', 'OUTSTANDINGP', 'LASTBILLEDDATE1',
       'LASTBILLEDDATE2', 'LASTBILLEDDATEP', 'HEALTHCARECLAIMTYPEID1',
       'HEALTHCARECLAIMTYPEID2', 'patient_id_pat', 'BIRTHDATE', 'DEATHDATE',
       'SSN', 'DRIVERS', 'PASSPORT', 'PREFIX', 'FIRST', 'MIDDLE', 'LAST',
       'SUFFIX', 'MAIDEN', 'MARITAL', 'RACE', 'ETHNICITY', 'GENDER',
       'BIRTHPLACE', 'ADDRESS', 'CITY', 'STATE', 'COUNTY', 'FIPS', 'ZIP',
       'LAT', 'LON', 'HEALTHCARE_EXPENSES', 'HEALTHCARE_COVERAGE', 'INCOME'],
      dtype='object')


,Id,patient_id,PROVIDERID,PRIMARYPATIENTINSURANCEID,SECONDARYPATIENTINSURANCEID,DEPARTMENTID,PATIENTDEPARTMENTID,DIAGNOSIS1,DIAGNOSIS2,DIAGNOSIS3,...,CITY,STATE,COUNTY,FIPS,ZIP,LAT,LON,HEALTHCARE_EXPENSES,HEALTHCARE_COVERAGE,INCOME
0,146ca7c8-be75-f1bc-c4a2-a8846438822d,146ca7c8-be75-f1bc-9e9b-c56a5caa5842,e132fe0c-83d9-3f9e-a6c8-5644c4f05853,26aab0cd-6aba-3e1b-ac5b-05c8867e762c,NaN,20,20,224299000,160904001.0,NaN,...,East Longmeadow,Massachusetts,Hampden County,NaN,0,42.028287,-72.497059,74899.55,10475.6,876903
1,146ca7c8-be75-f1bc-97c4-ba05acf233a0,146ca7c8-be75-f1bc-9e9b-c56a5caa5842,e132fe0c-83d9-3f9e-a6c8-5644c4f05853,26aab0cd-6aba-3e1b-ac5b-05c8867e762c,NaN,20,20,73595000,NaN,NaN,...,East Longmeadow,Massachusetts,Hampden County,NaN,0,42.028287,-72.497059,74899.55,10475.6,876903
2,146ca7c8-be75-f1bc-e23e-543fbd1acbf4,146ca7c8-be75-f1bc-9e9b-c56a5caa5842,e132fe0c-83d9-3f9e-a6c8-5644c4f05853,26aab0cd-6aba-3e1b-ac5b-05c8867e762c,NaN,20,20,314529007,160903007.0,422650009.0,...,East Longmeadow,Massachusetts,Hampden County,NaN,0,42.028287,-72.497059,74899.55,10475.6,876903
3,146ca7c8-be75-f1bc-6bdd-e07a3cda3c5e,146ca7c8-be75-f1bc-9e9b-c56a5caa5842,44dac0c7-a65c-388b-9ed5-a176a0567149,26aab0cd-6aba-3e1b-ac5b-05c8867e762c,NaN,1,1,444814009,444814009.0,NaN,...,East Longmeadow,Massachusetts,Hampden County,NaN,0,42.028287,-72.497059,74899.55,10475.6,876903
4,146ca7c8-be75-f1bc-1fe5-7ec83927fd0e,146ca7c8-be75-f1bc-9e9b-c56a5caa5842,e132fe0c-83d9-3f9e-a6c8-5644c4f05853,26aab0cd-6aba-3e1b-ac5b-05c8867e762c,NaN,20,20,314529007,160904001.0,NaN,...,East Longmeadow,Massachusetts,Hampden County,NaN,0,42.028287,-72.497059,74899.55,10475.6,876903


## 3. Clean patient demographics
6: Clean patient demographics
Start with the patients table as my base table.

Tasks:

inspect missing values
standardize column names
calculate age if DOB exists
keep only relevant columns

In [126]:
patients.columns = patients.columns.str.lower().str.strip()
patients.head()

,patient_id_pat,birthdate,deathdate,ssn,drivers,passport,prefix,first,middle,last,...,city,state,county,fips,zip,lat,lon,healthcare_expenses,healthcare_coverage,income
0,146ca7c8-be75-f1bc-9e9b-c56a5caa5842,1995-12-30,NaN,999-80-9624,S99917643,X66341700X,Mr.,Phil587,NaN,Pacocha935,...,East Longmeadow,Massachusetts,Hampden County,NaN,0,42.028287,-72.497059,74899.55,10475.60,876903
1,5b0065a1-f616-5871-5939-075d1006d277,1973-02-23,NaN,999-81-7925,S99986208,X49312881X,Mr.,Norberto865,NaN,Paucek755,...,Malden,Massachusetts,Middlesex County,25017.0,2155,42.454511,-71.124463,138979.05,55016.58,808564
2,498e42ed-7810-d8d5-bd1d-3b74900a6b60,2008-03-30,NaN,999-37-4769,S99943993,NaN,NaN,Frederic454,Connie24,Tillman293,...,Northampton,Massachusetts,Hampshire County,25015.0,1062,42.307627,-72.643857,62628.60,20839.84,27403
3,c405a320-49d4-df4d-3997-35cbea2aa90f,1991-09-06,NaN,999-28-2228,S99928098,X89582336X,Mrs.,Yer774,Ruby458,Hammes673,...,Shrewsbury,Massachusetts,Worcester County,NaN,0,42.300428,-71.758396,193696.50,294500.10,66272
4,26f41c1a-c3e6-cfa7-dbd7-64d442d1aef8,2023-04-30,NaN,999-63-5951,NaN,NaN,NaN,Joey457,So684,Macejkovic424,...,Ludlow,Massachusetts,Hampden County,NaN,0,42.225858,-72.486772,14201.62,0.00,50353


In [127]:
#keeping rel. columns only 
patients["birthdate"] = pd.to_datetime(patients["birthdate"], errors="coerce")
reference_date = pd.Timestamp("2026-01-01")
patients["age"] = ((reference_date - patients["birthdate"]).dt.days / 365.25).astype("float")

In [128]:
#filtering geriatric population
patients = patients[patients["age"] >= 65].copy()



In [129]:
#cleaning encounters table
encounters.columns = encounters.columns.str.lower().str.strip()

# Convert dates
encounters["start"] = pd.to_datetime(encounters["start"], errors="coerce")
encounters["stop"] = pd.to_datetime(encounters["stop"], errors="coerce")

# Drop irrelevant columns if needed
encounters = encounters[["encounter_id", "patient_id_enc", "start", "stop", "encounterclass", "code", "description"]]

# Optional: flag hospitalizations
encounters["is_hospitalization"] = encounters["encounterclass"].str.contains("inpatient", case=False, na=False)


In [130]:
#cleaning medications table
medications.columns = medications.columns.str.lower().str.strip()

# Convert dates
medications["start"] = pd.to_datetime(medications["start"], errors="coerce")
medications["stop"] = pd.to_datetime(medications["stop"], errors="coerce")
# Keep relevant columns
medications = medications[[
    "encounter_id",
    "patient_id_med",
    "code",
    "description",
    "base_cost",
    "totalcost"
]]



In [131]:
medications.columns.tolist()


['encounter_id',
 'patient_id_med',
 'code',
 'description',
 'base_cost',
 'totalcost']

In [132]:
conditions.columns.tolist()


['START',
 'STOP',
 'patient_id',
 'encounter_id',
 'SYSTEM',
 'CODE',
 'DESCRIPTION']

In [133]:
observations.columns.tolist()

['DATE',
 'patient_id',
 'encounter_id',
 'CATEGORY',
 'CODE',
 'DESCRIPTION',
 'VALUE',
 'UNITS',
 'TYPE']

In [134]:
claims.columns.tolist()

['Id',
 'patient_id',
 'PROVIDERID',
 'PRIMARYPATIENTINSURANCEID',
 'SECONDARYPATIENTINSURANCEID',
 'DEPARTMENTID',
 'PATIENTDEPARTMENTID',
 'DIAGNOSIS1',
 'DIAGNOSIS2',
 'DIAGNOSIS3',
 'DIAGNOSIS4',
 'DIAGNOSIS5',
 'DIAGNOSIS6',
 'DIAGNOSIS7',
 'DIAGNOSIS8',
 'REFERRINGPROVIDERID',
 'APPOINTMENTID',
 'CURRENTILLNESSDATE',
 'SERVICEDATE',
 'SUPERVISINGPROVIDERID',
 'STATUS1',
 'STATUS2',
 'STATUSP',
 'OUTSTANDING1',
 'OUTSTANDING2',
 'OUTSTANDINGP',
 'LASTBILLEDDATE1',
 'LASTBILLEDDATE2',
 'LASTBILLEDDATEP',
 'HEALTHCARECLAIMTYPEID1',
 'HEALTHCARECLAIMTYPEID2']

In [135]:
#cleaning CONDITIONS table
# Standardize column names
conditions.columns = conditions.columns.str.lower().str.strip()

# Convert dates
conditions["start"] = pd.to_datetime(conditions["start"], errors="coerce")
conditions["stop"] = pd.to_datetime(conditions["stop"], errors="coerce")

# Keep relevant columns
conditions = conditions[[
    "encounter_id",
    "patient_id",
    "system",
    "code",
    "description",
    "start",
    "stop"
]]

# Optional: flag chronic conditions
chronic_keywords = ["diabetes", "heart", "renal", "copd", "cancer", "stroke"]
conditions["is_chronic"] = conditions["description"].str.lower().str.contains("|".join(chronic_keywords), na=False)


In [136]:
#clean observations table
# Standardize column names
observations.columns = observations.columns.str.lower().str.strip()

# Convert date
observations["date"] = pd.to_datetime(observations["date"], errors="coerce")

# Keep relevant columns
observations = observations[[
    "encounter_id",
    "patient_id",
    "date",
    "category",
    "code",
    "description",
    "value",
    "units",
    "type"
]]

# Optional: example abnormal flag (BMI > 30)
observations["is_abnormal_bmi"] = (
    observations["description"].str.contains("bmi", case=False, na=False) &
    (pd.to_numeric(observations["value"], errors="coerce") > 30)
)


In [137]:
#Clean CLAIMS Table
# Standardize column names
claims.columns = claims.columns.str.lower().str.strip()

# Convert date columns
date_cols = [
    "currentillnessdate", "servicedate",
    "lastbilleddate1", "lastbilleddate2", "lastbilleddatep"
]

for col in date_cols:
    if col in claims.columns:
        claims[col] = pd.to_datetime(claims[col], errors="coerce")

# Keep relevant columns
claims = claims[[
    "id",
    "patient_id",
    "providerid",
    "primarypatientinsuranceid",
    "secondarypatientinsuranceid",
    "departmentid",
    "patientdepartmentid",
    "diagnosis1", "diagnosis2", "diagnosis3", "diagnosis4",
    "diagnosis5", "diagnosis6", "diagnosis7", "diagnosis8",
    "currentillnessdate",
    "servicedate",
    "status1", "status2", "statusp",
    "outstanding1", "outstanding2", "outstandingp"
]]

# Derived features
diag_cols = [f"diagnosis{i}" for i in range(1, 9)]
claims["diagnosis_count"] = claims[diag_cols].notna().sum(axis=1)

claims["total_outstanding"] = claims[["outstanding1", "outstanding2", "outstandingp"]].sum(axis=1)



## 4. Create patient-level summary tables
7: Build patient-level summary tables from each file

summarize each table to patient level first.

A. Encounters summary

Create features like:
total encounters
encounter type counts
hospitalization count if possible

In [139]:
encounters.columns.tolist()


['encounter_id',
 'patient_id_enc',
 'start',
 'stop',
 'encounterclass',
 'code',
 'description',
 'is_hospitalization']

#Mapping real column names to do this step

In [140]:
#encounters table
encounters["patient"] = encounters["patient_id_enc"]
encounters["id"] = encounters["encounter_id"]


In [141]:
#conditions table
conditions["patient"] = conditions["patient_id"]


In [142]:
#Medications table
medications.loc[:, "patient"] = medications["patient_id_med"]


In [143]:
#Observations table
observations["patient"] = observations["patient_id"]

In [144]:
#claims table
claims["patient"] = claims["patient_id"]


In [145]:
#encounters summary
encounters.columns = encounters.columns.str.lower().str.strip()

encounter_summary = encounters.groupby("patient").agg(
    total_encounters=("id", "count")
).reset_index()

encounters["encounterclass"] = encounters["encounterclass"].fillna("unknown")

hospitalizations = (
    encounters.assign(is_hospitalization=encounters["encounterclass"].str.contains("inpatient", case=False, na=False))
    .groupby("patient")["is_hospitalization"]
    .sum()
    .reset_index(name="hospitalization_count")
)

encounter_summary = encounter_summary.merge(hospitalizations, on="patient", how="left")


In [146]:
#Conditions summary
conditions.columns = conditions.columns.str.lower().str.strip()

condition_summary = conditions.groupby("patient").agg(
    total_conditions=("code", "count"),
    unique_conditions=("description", "nunique")
).reset_index()


In [147]:
# medications summary
medications.columns = medications.columns.str.lower().str.strip()

medication_summary = medications.groupby("patient").agg(
    total_medications=("code", "count"),
    unique_medications=("description", "nunique")
).reset_index()


In [148]:
#Claims summary
claims.columns = claims.columns.str.lower().str.strip()

claims_summary = claims.groupby("patient").agg(
    total_claims=("id", "count")
).reset_index()


In [149]:
#Observations summary
observations.columns = observations.columns.str.lower().str.strip()

observation_summary = observations.groupby("patient").agg(
    total_observations=("description", "count")
).reset_index()


## 5. Merge summaries into one analytics dataset
8: Merge summaries into one patient-level table

Using the patient table as the base.

In [150]:
#Add id column to patients
patients["id"] = patients["patient_id_pat"]   # or whatever your patient ID column is


In [151]:
#Merge all summaries
patient_level = patients.copy()

patient_level = patient_level.merge(encounter_summary, left_on="id", right_on="patient", how="left")
patient_level = patient_level.merge(condition_summary, left_on="id", right_on="patient", how="left", suffixes=("", "_cond"))
patient_level = patient_level.merge(medication_summary, left_on="id", right_on="patient", how="left", suffixes=("", "_med"))
patient_level = patient_level.merge(claims_summary, left_on="id", right_on="patient", how="left", suffixes=("", "_claim"))
patient_level = patient_level.merge(observation_summary, left_on="id", right_on="patient", how="left", suffixes=("", "_obs"))


In [152]:
#Clean up duplicate patient columns: after merging, we have multiple patient columns from the right-hand tables. Clean them up:
patient_level = patient_level.drop(columns=[
    "patient", "patient_cond", "patient_med", "patient_claim", "patient_obs"
], errors="ignore")


9: Fill missing values

Patients with no record in one table should not stay null for count columns.

In [153]:
count_cols = [
    "total_encounters",
    "hospitalization_count",
    "total_conditions",
    "unique_conditions",
    "total_medications",
    "unique_medications",
    "total_claims",
    "total_observations"
]

for col in count_cols:
    if col in patient_level.columns:
        patient_level[col] = patient_level[col].fillna(0)


## 6. Engineer first-pass care gap features
10: Create first-pass analytics features

This is where the project starts becoming interesting.

Create:

age group
high utilization flag
high comorbidity flag
possible care gap proxy

In [154]:
#creating clean, interpretable geriatric segments
patient_level["age_group"] = pd.cut(
    patient_level["age"],
    bins=[65, 75, 85, 120],
    labels=["65-74", "75-84", "85+"],
    right=False
)

In [155]:
#High Utilization Flag :identifies patients in the top 25% of encounter volume:: High utilizers often drive cost, risk, and care coordination needs.
patient_level["high_utilization_flag"] = (
    patient_level["total_encounters"] > patient_level["total_encounters"].quantile(0.75)
).astype(int)


In [156]:
#High Comorbidity Flag:  uses the unique condition count to identify medically complex patients:powerful proxy for clinical burden.
patient_level["high_comorbidity_flag"] = (
    patient_level["unique_conditions"] > patient_level["unique_conditions"].quantile(0.75)
).astype(int)


In [157]:
#Potential Care Gap Flag:high clinical burden + low follow-up.
patient_level["potential_care_gap_flag"] = (
    (patient_level["high_comorbidity_flag"] == 1) &
    (patient_level["total_encounters"] < patient_level["total_encounters"].median())
).astype(int)


Interpretation:

High comorbidity

But fewer encounters than expected

→ Possible underutilization or unmet need

This was a first-pass care gap signal, and you can refine it later with:

preventive care adherence

follow-up gaps

chronic disease monitoring

medication adherence

risk scores

In [158]:
11: Check the final table

SyntaxError: illegal target for annotation (2554220326.py, line 1)

In [159]:
#Check the final table
patient_level.head()
patient_level.shape
patient_level.info()
patient_level.describe(include="all")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31 entries, 0 to 30
Data columns (total 42 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   patient_id_pat           31 non-null     object        
 1   birthdate                31 non-null     datetime64[ns]
 2   deathdate                13 non-null     object        
 3   ssn                      31 non-null     object        
 4   drivers                  30 non-null     object        
 5   passport                 30 non-null     object        
 6   prefix                   30 non-null     object        
 7   first                    31 non-null     object        
 8   middle                   25 non-null     object        
 9   last                     31 non-null     object        
 10  suffix                   0 non-null      object        
 11  maiden                   5 non-null      object        
 12  marital                  30 non-null  

,patient_id_pat,birthdate,deathdate,ssn,drivers,passport,prefix,first,middle,last,...,total_conditions,unique_conditions,total_medications,unique_medications,total_claims,total_observations,age_group,high_utilization_flag,high_comorbidity_flag,potential_care_gap_flag
count,31,31,13,31,30,30,30,31,25,31,...,31.000000,31.000000,31.000000,31.000000,31.000000,31.000000,31,31.000000,31.000000,31.0
unique,31,NaN,13,31,30,30,2,31,25,31,...,NaN,NaN,NaN,NaN,NaN,NaN,3,NaN,NaN,NaN
top,cd1812c1-21c3-e70e-c635-fec819c488aa,NaN,1989-06-12,999-71-5044,S99951898,X62372693X,Mr.,Marquetta382,Maryetta775,Cole117,...,NaN,NaN,NaN,NaN,NaN,NaN,65-74,NaN,NaN,NaN
freq,1,NaN,1,1,1,1,25,1,1,1,...,NaN,NaN,NaN,NaN,NaN,NaN,14,NaN,NaN,NaN
mean,NaN,1940-02-11 06:11:36.774193536,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,60.193548,26.903226,142.193548,11.290323,229.225806,1817.612903,NaN,0.258065,0.258065,0.0
min,NaN,1916-03-24 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,8.000000,5.000000,4.000000,3.000000,23.000000,137.000000,NaN,0.000000,0.000000,0.0
25%,NaN,1921-12-26 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,32.500000,22.000000,18.500000,5.000000,57.500000,565.000000,NaN,0.000000,0.000000,0.0
50%,NaN,1948-10-27 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,40.000000,26.000000,48.000000,10.000000,93.000000,754.000000,NaN,0.000000,0.000000,0.0
75%,NaN,1955-06-14 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,48.000000,30.500000,121.500000,16.000000,233.500000,1819.500000,NaN,0.500000,0.500000,0.0
max,NaN,1960-02-22 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,270.000000,52.000000,1132.000000,29.000000,1559.000000,12651.000000,NaN,1.000000,1.000000,0.0


the previous table confirms:
one row per patient
no obvious duplicate patients
features look sensible

Check duplicates:

In [160]:
patient_level["id"].duplicated().sum()

np.int64(0)

## 7. Save patient-level dataset
12: Saving the final integrated dataset

In [161]:
OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

patient_level.to_csv(OUTPUT_DIR / "patient_level_analytics.csv", index=False)